In [46]:
from deepeval import evaluate 
from deepeval.metrics import faithfulness, AnswerRelevancyMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase
import sys
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Adds the parent directory to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

from app.rag import EmailRAG


In [47]:
# 1. Get current directory (likely .../email_rag/evals)
cwd = os.getcwd()

# 2. If you are inside 'evals', move to the parent directory
if os.path.basename(cwd) == 'evals':
    os.chdir('..')

# 3. Add the new root to sys.path so imports work
sys.path.append(os.getcwd())

print(f"✅ Now working in: {os.getcwd()}")

✅ Now working in: /Users/shafeela/Documents/AI_projects/email_rag


In [48]:
db_file ='app/chroma_db/chroma.sqlite3'
if os.path.exists(db_file):
    print("✅ Database file found!")
    rag = EmailRAG()
else:
    print(f"❌ Cannot find database at {os.path.abspath(db_file)}")

✅ Database file found!


In [49]:
query = "when is the badminton tryout?"
response = answer = rag.ask(query)

print(f'response : {response}')

response : The badminton tryout is on **Tuesday, 2/3** from **5:30 PM-8:30 PM** for Returning Players and **6:00PM-8:30 PM** in the **Main Gym** for Experienced Players who are new to the team.


In [50]:
import chromadb
path_to_db = os.path.abspath("app/chroma_db")
client = chromadb.PersistentClient(path=path_to_db)
collection = client.get_collection("email_rag_nomic")

# 2. Get the actual text
results = collection.get(include=["documents"])
texts = results.get("documents", [])
if texts:
    print(f"✅ Found {len(texts)} chunks. Starting synthesis...")

✅ Found 293 chunks. Starting synthesis...


In [62]:
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(
        model='deepseek-r1:8b',
        temperature=0,
        base_url="http://192.168.1.75:11434"
        )
try:
    messages = [
    SystemMessage(content="You are a helpful assistant. You must respond ONLY in English. Do not use Chinese characters even in your internal thoughts."),
    HumanMessage(content="hi")
]
    response = llm.invoke(messages)
    print(response.content)
except Exception as e:
    print(f"❌ Connection failed: {e}")

Hello! How can I assist you today?


In [51]:
docs = rag.get_random_chunk(num_chunks=5)
print(docs)
contexts = [[doc[:600]]for doc in docs]
print(contexts)


['GameChanger RSVP for the Leland Varsity Chargers game Event Name Away Game vs. Evergreen Valley Tournament When Sat, December 13 Arrive 7:00 AM, 7:00 AM - 1:00 PM PT Where Evergreen Valley High School 3300 Quimby Rd San Jose CA 95148 Notes No Notes RSVP Leland Varsity Chargers is using the free GameChanger app for scheduling and team announcements. Unsubscribe 124 East 14th Street, 17th Floor, New York, NY 10003 © GameChanger Media, Inc. A DICK’S Sporting Goods Company', 'GameChanger RSVP for the Leland Varsity Chargers game Event Name Away Game vs. Trojan War When Sat, January 31 Arrive 7:00 AM, 7:00 AM - 8:00 AM PT Where Milpitas High School 1285 Escuela Pkwy Milpitas CA 95035 Notes No Notes RSVP Leland Varsity Chargers is using the free GameChanger app for scheduling and team announcements. Unsubscribe 124 East 14th Street, 17th Floor, New York, NY 10003 © GameChanger Media, Inc. A DICK’S Sporting Goods Company', 'we will be reviewing our Comprehensive School Safety Plan (CSSP) - 

In [52]:
from deepeval.synthesizer import Synthesizer
from deepeval.dataset import EvaluationDataset
from deepeval.models import OllamaModel 

custom_model = OllamaModel(
    model='llama3.1',
    base_url='http://192.168.1.75:11434'
)

In [ ]:
from deepeval import synthesizer
from deepeval.synthesizer.config import StylingConfig



custom_model = OllamaModel(
    model='llama3.1',
    base_url='http://192.168.1.75:11434'
)

styling_config = StylingConfig(
    task=(
    "Generate a single, direct, and ultra-short question (max 10 words) based on the provided email context. "
    "The question should be simple and specific, such as 'When is the badminton tryout?' or 'Who is the coach?'."
),
input_format=(
    "A simple, one-sentence question ending in a question mark. "
    "Strictly no conversational filler, introductory text, or complex multi-part questions."
),
scenario="A student is quickly asking a school administrator for a specific detail via email."
)
synthesizer = Synthesizer(
    model=custom_model,
    styling_config=styling_config
)

goldens = synthesizer.generate_goldens_from_contexts(
    contexts = contexts,
    include_expected_output = True,
    max_goldens_per_context = 1
)

In [53]:
# dataset = EvaluationDataset(goldens=goldens)
# dataset.save_as(file_type="json", directory="./data", file_name="email_goldens")

dataset = EvaluationDataset()
goldens = dataset.add_goldens_from_json_file(
    file_path="./data/email_goldens.json"
)
# print(f"✅ Success! Generated {len(goldens)} golden pairs.")

In [54]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

from deepeval import evaluate

COMMON_JUDGE_RULES = """
- IGNORE all formatting differences (e.g., bullet points vs. paragraphs).
- IGNORE date format variations (e.g., '2/3' is identical to 'February 3rd').
- IGNORE the presence or absence of the year if the month and day are correct.
- PRIORITIZE factual agreement over word-for-word matching.
"""


judge_model = custom_model

faithfullness = FaithfulnessMetric(threshold=0.5, model=judge_model, include_reason=True)
relevancy = AnswerRelevancyMetric(threshold=0.5, model=judge_model, include_reason=True)
precision = ContextualPrecisionMetric(threshold=0.5, model=judge_model, include_reason=True)
recall_metric = ContextualRecallMetric(threshold=0.5, model=judge_model, include_reason=True)

correctness_metric = GEval(
    name ="correctness",
    evaluation_params= [
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    model=judge_model,  
    threshold=0.7,
    criteria=f"Determine if the outputs are factually identical. Rules: {COMMON_JUDGE_RULES}"

)
test_cases= []

for golden in dataset.goldens:
    response = rag.ask(golden.input)
   
    print(f'golden.input : {golden.input}')
    print(f'golden.expected_output : {golden.expected_output}')
    print(f'golden.context : {golden.context}')
    print(f'response : {response}')
    docs = rag.get_relevant_doc(golden.input,2)
    print(f'docs from get_relevant_doc :{docs}')
    test_case = LLMTestCase(
        input = golden.input,
        actual_output = response,
        retrieval_context=docs,
        expected_output=golden.expected_output,
        context = golden.context
    )
    test_cases.append(test_case)


print("📊 Starting Evaluation...")
evaluate(test_cases, [faithfullness, relevancy, precision, recall_metric])
correctness_metric.measure(test_case)

golden.input : When is the Digital Safety Presentation scheduled for December?
golden.expected_output : The Digital Safety Presentation is scheduled for Wednesday, December 10, 2025 at 8:15 a.m.
golden.context : ["Digital Safety ** This month's Principal's Chat will be presented by Casey Vess, a Crime Prevention Specialist from SJPD on the topic of Digital Safety. Graystone will learn about best practices to help our students use the internet responsibly. Student presentations will also be provided by SJPD in December. Please RSVP on the Digital Safety Presentation post so we can prepare for the appropriate amount of attendees. We hope you join us on **Wednesday, December 10, 2025 at 8:15 a.m**. Light refreshments will be provided. ** School Site Council ** The 3rd School Site Council will be on **Wedne"]
response : The Digital Safety Presentation is scheduled for **December 10th** from **3:00pm-4:00pm** in the Leland Main Office.
docs from get_relevant_doc :["1. PRINCIPAL'S UPDATE 10/

✨ You're running DeepEval's latest Faithfulness Metric! (using llama3.1 (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.1 (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using llama3.1 (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using llama3.1 (Ollama), strict=False, 
async_mode=True)...

TimeoutError: 

In [58]:
# Try an 'Easy' search for the missing document
test_docs = rag.vectorstore.similarity_search("Digital Safety Presentation", k=3)

if not any("Digital Safety" in doc.page_content for doc in test_docs):
    print("❌ CRITICAL: The document is NOT in the database. Re-index your source files!")
else:
    print("✅ Document found in DB. The issue was the 'similarity' ranking.")

❌ CRITICAL: The document is NOT in the database. Re-index your source files!
